# 🌿 CSIRO Image2Biomass: Dual-Stream DINO Vision Pipeline
### Unified 5-Fold Training Notebook with Balanced Stratification & Interval Classification

This notebook integrates the core winning techniques from the **1st, 2nd, and 3rd place solutions**:
1. **Balanced Stratification**: Joint StratifiedKFold on `State` + Binned Composite Biomass to eliminate fold imbalance.
2. **Dual-Stream Input**: Splits panoramic 2000×1000 image into Left & Right 1:1 square views with independent augmentations.
3. **DINO ViT Backbone**: High-capacity Vision Transformer (`vit_base_patch14_dinov2` or `vit_base_patch16_dinov3_qkvb`) with cross-view self-attention.
4. **5 Direct Regression Heads**: Directly predicts Green, Dead, Clover, GDM, and Total (no compounding summation errors).
5. **5 Auxiliary Interval Classification Heads**: 7 non-uniform density intervals (UEPNet formulation, +0.03 LB/PB boost).
6. **Dual-Objective Loss**: Weighted SmoothL1 + CrossEntropy weighted by official competition metric: `[0.1, 0.1, 0.1, 0.2, 0.5]`.
7. **2-Stage Training Schedule**: Stage 1 head warm-up (backbone frozen) -> Stage 2 end-to-end fine-tuning with differential LR.

In [ ]:
# 1. Environment & Hardware Verification
import os
import sys
import time
import math
import random
import glob
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
print("✓ Environment initialized with deterministic seeds.")

## 2. Configuration & Hyperparameters
All hyperparameters are centralized here for easy tuning.

In [ ]:
class Config:
    # Data
    data_path = 'train_converted.csv' if os.path.exists('train_converted.csv') else 'wide.csv'
    img_size = 512          # 512 or 1024 for single square sub-image
    n_folds = 5             # 5 folds guarantees stable CV-LB correlation
    
    # Model
    backbone = 'vit_base_patch14_dinov2'  # or 'vit_base_patch16_dinov3_qkvb'
    fusion_dim = 384
    dropout = 0.3
    
    # Optimization
    batch_size = 8
    grad_accum_steps = 2    # Effective batch size = 16
    lr = 3e-4               # Base learning rate for heads
    backbone_lr_ratio = 0.1 # 3e-5 for pre-trained backbone
    weight_decay = 0.01
    
    # Schedule
    stage1_epochs = 8       # Warm-up heads with frozen backbone
    stage2_epochs = 26      # Full end-to-end fine-tuning
    warmup_epochs = 3       # Linear warmup for Stage 2
    
    # Targets & Weights
    target_names = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    official_weights = [0.1, 0.1, 0.1, 0.2, 0.5]
    cls_weight = 0.3        # Auxiliary classification loss weight
    
    output_dir = 'models'
    seed = 42

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
print(f"✓ Configured: {cfg.backbone} | Resolution: {cfg.img_size}x{cfg.img_size} | {cfg.n_folds} Folds")

## 3. Balanced Stratification (Addressing the Stratification Problem)

**The Problem with Date Grouping:**
With only 357 images across 4 States (NSW: 75, Tas: 138, Vic: 112, WA: 32) and only 28 dates, WA has only 3 sampling dates! Grouping strictly by date isolates entire states and rare species in single folds, causing massive label distribution shifts between train and val (leading to negative/poor validation R²).

**The Solution (from 2nd & 3rd place):**
We construct a composite weighted biomass label (`0.1*Green + 0.1*Dead + 0.1*Clover + 0.2*GDM + 0.5*Total`) and discretize it into quantiles. We then stratify by `State + Biomass_Bin`. This ensures every fold has:
- Equal sample count (~71-72 samples)
- Equal state distribution
- Balanced mean and variance across all 5 targets

In [ ]:
def create_balanced_stratified_folds(df, n_splits=5, seed=42):
    df = df.copy().reset_index(drop=True)
    composite = sum(df[t] * w for t, w in zip(cfg.target_names, cfg.official_weights))
    try:
        biomass_bins = pd.qcut(composite, q=5, labels=False, duplicates='drop')
    except ValueError:
        biomass_bins = pd.qcut(composite, q=3, labels=False, duplicates='drop')
    
    strat_key = df['State'].astype(str) + '_' + biomass_bins.astype(str)
    counts = strat_key.value_counts()
    rare = counts[counts < n_splits].index
    strat_key = strat_key.apply(lambda k: k.split('_')[0] + '_other' if k in rare else k)
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    df['fold'] = -1
    for fold_idx, (tr, va) in enumerate(skf.split(df, strat_key)):
        df.loc[va, 'fold'] = fold_idx
    return df

df = pd.read_csv(cfg.data_path)
df = create_balanced_stratified_folds(df, n_splits=cfg.n_folds, seed=cfg.seed)

print(f"Loaded {len(df)} samples. Verification of Fold Balance:")
for f in range(cfg.n_folds):
    sub = df[df['fold'] == f]
    st_dict = dict(sub['State'].value_counts())
    tot_mean = sub['Dry_Total_g'].mean()
    tot_std = sub['Dry_Total_g'].std()
    print(f"  Fold {f+1}: N={len(sub):2d} | States={st_dict} | Total Biomass Mean={tot_mean:.1f}g (std={tot_std:.1f})")

## 4. Augmentation Pipeline & Dual-Stream Dataset
- **Camera Scaling Simulation**: Downscale 0.85-1.0 and pad with black pixels (1st place, simulates drone/camera focal height variation).
- **Vertical Strip Permutation**: Shuffles 4 vertical slices (3rd place, conserves total grams while breaking memorization).
- **CLAHE & Gaussian Noise**: Local contrast equalization and sensor noise.
- **Left/Right View Swap**: Randomly swaps the left and right plot halves (50% probability).
- **UEPNet 7-Interval Discretization**: Divides biomass into 7 coarse bins for auxiliary classification.

In [ ]:
BORDERS_DICT = {
    'Dry_Green_g':  [1.6e-05, 13.4232, 27.0782, 45.5236, 79.834, 157.9836],
    'Dry_Dead_g':   [1.6e-05, 6.1407, 13.1192, 23.277, 38.8581, 83.8407],
    'Dry_Clover_g': [1.6e-05, 3.9, 10.5353, 20.6523, 37.5911, 71.7865],
    'GDM_g':        [1.6e-05, 16.5143, 30.507, 49.5585, 81.0, 157.9836],
    'Dry_Total_g':  [1.6e-05, 23.4907, 41.1, 61.1, 96.8288, 185.7],
}

def get_interval_labels(targets_np):
    labels = np.zeros_like(targets_np, dtype=np.int64)
    for col_idx, col_name in enumerate(cfg.target_names):
        borders = BORDERS_DICT.get(col_name)
        if borders is not None:
            labels[:, col_idx] = np.digitize(targets_np[:, col_idx], borders)
        else:
            labels[:, col_idx] = np.clip(np.digitize(targets_np[:, col_idx], [0, 5, 15, 30, 60, 120]), 0, 6)
    return labels

def apply_camera_scaling(img, prob=0.2):
    if random.random() < prob:
        h, w = img.shape[:2]
        bg = np.zeros_like(img)
        scale = random.uniform(0.85, 1.0)
        nw, nh = max(1, int(w * scale)), max(1, int(h * scale))
        resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_CUBIC)
        top = random.randint(0, h - nh)
        left = random.randint(0, w - nw)
        bg[top:top + nh, left:left + nw] = resized
        return bg
    return img

def apply_vertical_strip_shuffle(img, n_strips=4, prob=0.3):
    if random.random() < prob:
        strips = np.array_split(img, n_strips, axis=1)
        random.shuffle(strips)
        return np.concatenate(strips, axis=1)
    return img

class DualStreamBiomassDataset(Dataset):
    def __init__(self, df, img_size=512, is_training=True):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.is_training = is_training
        
        if self.is_training:
            self.pil_transform = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.RandomApply([transforms.RandomRotation((90, 90))], p=0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                transforms.RandomGrayscale(p=0.15),
                transforms.ToTensor(),
                transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])
        else:
            self.pil_transform = transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
            ])
        
        self.has_targets = all(t in self.df.columns for t in cfg.target_names)
        if self.has_targets:
            self.targets_reg = self.df[cfg.target_names].values.astype(np.float32)
            self.targets_cls = get_interval_labels(self.targets_reg)

    def __len__(self):
        return len(self.df)

    def _resolve_image_path(self, rel_path):
        if os.path.exists(rel_path):
            return rel_path
        fname = os.path.basename(rel_path)
        for cand in ['train', 'test', 'images', os.path.join('..', 'train')]:
            p = os.path.join(cand, fname)
            if os.path.exists(p):
                return p
        raise FileNotFoundError(f"Cannot find image: {rel_path}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_image_path(row['image_path'])
        raw_bgr = cv2.imread(img_path)
        raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
        h, w, _ = raw_rgb.shape
        mid_w = w // 2
        
        left_np = raw_rgb[:, :mid_w].copy()
        right_np = raw_rgb[:, mid_w:].copy()
        
        if self.is_training:
            if random.random() < 0.5:
                left_np, right_np = right_np, left_np
            left_np = apply_camera_scaling(left_np, prob=0.2)
            right_np = apply_camera_scaling(right_np, prob=0.2)
            left_np = apply_vertical_strip_shuffle(left_np, prob=0.3)
            right_np = apply_vertical_strip_shuffle(right_np, prob=0.3)
        
        left_tensor = self.pil_transform(Image.fromarray(left_np))
        right_tensor = self.pil_transform(Image.fromarray(right_np))
        
        item = {
            'image_left': left_tensor,
            'image_right': right_tensor,
            'sample_id': row.get('sample_id', f'sample_{idx}'),
            'state': row.get('State', 'Unknown')
        }
        if self.has_targets:
            item['targets_reg'] = torch.tensor(self.targets_reg[idx], dtype=torch.float32)
            item['targets_cls'] = torch.tensor(self.targets_cls[idx], dtype=torch.long)
        return item

print("✓ Augmentation pipeline and DualStreamBiomassDataset defined.")

## 5. Model Architecture: Dual-Stream DINO ViT

```
Left Image [B, 3, 512, 512]  ---> [ Shared DINO ViT Backbone ] ---> [B, 768]
                                                                          \ 
                                                                           ---> [ Cross-View Multi-Head Attention ] ---> [ Fusion MLP ]
                                                                          /                                                   |
Right Image [B, 3, 512, 512] ---> [ Shared DINO ViT Backbone ] ---> [B, 768]                                                 |
                                                                                                   +--------------------------+
                                                                                                   |                          |
                                                                                         [ 5 Regression Heads ]     [ 5 Classification Heads ]
                                                                                         (Green, Dead, Clover,      (7-bin interval classes)
                                                                                          GDM, Total in grams)         auxiliary loss 0.3
```

In [ ]:
class DualStreamDINO(nn.Module):
    def __init__(self, backbone_name="vit_base_patch14_dinov2", fusion_dim=384, dropout=0.3, pretrained=True):
        super().__init__()
        self.backbone_name = backbone_name
        self.fusion_dim = fusion_dim
        self.num_targets = 5
        self.num_intervals = 7

        kwargs = {}
        if 'dinov2' in backbone_name or 'patch14' in backbone_name or 'patch16' in backbone_name:
            kwargs['dynamic_img_size'] = True

        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, **kwargs)
        self.backbone_dim = self.backbone.num_features

        n_heads = 8 if self.backbone_dim % 8 == 0 else 4
        self.cross_view_attn = nn.MultiheadAttention(
            embed_dim=self.backbone_dim, num_heads=n_heads, dropout=0.1, batch_first=True
        )
        self.attn_norm = nn.LayerNorm(self.backbone_dim)

        self.fusion_mlp = nn.Sequential(
            nn.Linear(self.backbone_dim * 2, self.fusion_dim),
            nn.LayerNorm(self.fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.reg_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.fusion_dim, self.fusion_dim // 2),
                nn.LayerNorm(self.fusion_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(self.fusion_dim // 2, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            ) for _ in range(self.num_targets)
        ])

        self.cls_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.fusion_dim, 128),
                nn.LayerNorm(128),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(128, self.num_intervals)
            ) for _ in range(self.num_targets)
        ])

    def extract_features(self, x):
        feats = self.backbone(x)
        if len(feats.shape) == 3:
            return feats.mean(dim=1)
        elif len(feats.shape) == 4:
            return feats.mean(dim=[2, 3])
        return feats

    def forward(self, img_left, img_right):
        feat_l = self.extract_features(img_left)
        feat_r = self.extract_features(img_right)
        tokens = torch.stack([feat_l, feat_r], dim=1)
        attn_out, _ = self.cross_view_attn(tokens, tokens, tokens)
        tokens = self.attn_norm(tokens + attn_out)
        fused = torch.cat([tokens[:, 0], tokens[:, 1]], dim=-1)
        fused = self.fusion_mlp(fused)
        reg_preds = [F.softplus(head(fused)) for head in self.reg_heads]
        cls_preds = [head(fused) for head in self.cls_heads]
        return reg_preds, cls_preds

print("✓ DualStreamDINO architecture instantiated successfully.")

## 6. Loss Function, Competition Metric & Post-Processing
- **Loss Function**: SmoothL1 for continuous regression + CrossEntropy for intervals, weighted by `[0.1, 0.1, 0.1, 0.2, 0.5]`.
- **Competition Metric**: Log-space R²: $\sum w_i \times R^2(\log(1+y_{true}), \log(1+y_{pred}))$.
- **Soft-Blend Post-Processing**: Clover scaling ($0.8\times$), Dead extreme correction ($>20\times 1.1, <10\times 0.9$), and soft blending of direct Total/GDM predictions with physical identities.

In [ ]:
class WeightedBiomassLoss(nn.Module):
    def __init__(self, cls_weight=0.3):
        super().__init__()
        self.criterion_reg = nn.SmoothL1Loss()
        self.criterion_cls = nn.CrossEntropyLoss()
        self.cls_weight = cls_weight
        self.weights = torch.tensor(cfg.official_weights, dtype=torch.float32)

    def forward(self, reg_preds, cls_preds, targets_reg, targets_cls=None):
        device = targets_reg.device
        w = self.weights.to(device)
        loss_reg = torch.tensor(0.0, device=device)
        for i in range(5):
            loss_reg += w[i] * self.criterion_reg(reg_preds[i].squeeze(-1), targets_reg[:, i])
        loss_cls = torch.tensor(0.0, device=device)
        if cls_preds is not None and targets_cls is not None:
            for i in range(5):
                loss_cls += w[i] * self.criterion_cls(cls_preds[i], targets_cls[:, i])
        return loss_reg + (self.cls_weight * loss_cls), loss_reg, loss_cls

def calculate_competition_r2(y_true, y_pred, weights=cfg.official_weights):
    yt = np.log1p(np.maximum(0, np.asarray(y_true, dtype=np.float64)))
    yp = np.log1p(np.maximum(0, np.asarray(y_pred, dtype=np.float64)))
    w = np.asarray(weights, dtype=np.float64)
    r2_scores = []
    for i in range(5):
        ss_res = np.sum((yt[:, i] - yp[:, i]) ** 2)
        ss_tot = np.sum((yt[:, i] - np.mean(yt[:, i])) ** 2)
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
        r2_scores.append(r2)
    return float(np.sum(w * np.array(r2_scores))), r2_scores

def apply_soft_blend_postprocess(preds_5, states=None):
    preds = np.maximum(np.asarray(preds_5, dtype=np.float32).copy(), 0.0)
    green = preds[:, 0]
    dead = preds[:, 1]
    clover = preds[:, 2] * 0.8
    gdm = preds[:, 3]
    total = preds[:, 4]
    
    dead = np.where(dead > 20.0, dead * 1.1, np.where(dead < 10.0, dead * 0.9, dead))
    if states is not None:
        for idx, st in enumerate(states):
            if str(st).strip() == 'WA':
                dead[idx] = 0.0
                
    gdm_blended = 0.5 * gdm + 0.5 * (green + clover)
    total_blended = 0.5 * total + 0.5 * (green + clover + dead)
    return np.maximum(np.column_stack([green, dead, clover, gdm_blended, total_blended]), 0.0)

print("✓ Loss function, competition metric, and soft physical post-processing defined.")

## 7. 2-Stage Training Engine
- **Stage 1 (Head Warmup)**: Freeze backbone for 8 epochs. Train only heads and cross-view attention with base LR (3e-4).
- **Stage 2 (Full End-to-End Fine-Tuning)**: Unfreeze backbone for 26 epochs. Backbone receives $0.1\times$ learning rate (3e-5) with 3 warmup epochs and Cosine Annealing.

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scaler, grad_accum=2):
    model.train()
    tot_loss, samples = 0.0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(loader):
        img_l = batch['image_left'].to(device)
        img_r = batch['image_right'].to(device)
        t_reg = batch['targets_reg'].to(device)
        t_cls = batch['targets_cls'].to(device)
        bs = img_l.size(0)
        
        with torch.amp.autocast('cuda'):
            reg_preds, cls_preds = model(img_l, img_r)
            loss, _, _ = criterion(reg_preds, cls_preds, t_reg, t_cls)
            loss = loss / grad_accum
            
        scaler.scale(loss).backward()
        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        tot_loss += loss.item() * grad_accum * bs
        samples += bs
    return tot_loss / max(1, samples)

def validate(model, loader, criterion, val_df, use_tta=True):
    model.eval()
    tot_loss, samples = 0.0, 0
    preds_list, targets_list = [], []
    with torch.no_grad():
        for batch in loader:
            img_l = batch['image_left'].to(device)
            img_r = batch['image_right'].to(device)
            t_reg = batch['targets_reg'].to(device)
            t_cls = batch['targets_cls'].to(device)
            bs = img_l.size(0)
            
            if use_tta:
                r1, c1 = model(img_l, img_r)
                r2, c2 = model(torch.flip(img_r, [3]), torch.flip(img_l, [3]))
                reg = [(a + b) * 0.5 for a, b in zip(r1, r2)]
                cls_p = [(a + b) * 0.5 for a, b in zip(c1, c2)]
            else:
                reg, cls_p = model(img_l, img_r)
                
            loss, _, _ = criterion(reg, cls_p, t_reg, t_cls)
            tot_loss += loss.item() * bs
            samples += bs
            preds_list.append(torch.cat(reg, dim=1).cpu().numpy())
            targets_list.append(t_reg.cpu().numpy())
            
    preds_raw = np.concatenate(preds_list, axis=0)
    targets_true = np.concatenate(targets_list, axis=0)
    states = val_df['State'].tolist() if 'State' in val_df.columns else None
    preds_post = apply_soft_blend_postprocess(preds_raw, states=states)
    r2_raw, _ = calculate_competition_r2(targets_true, preds_raw)
    r2_post, per_target = calculate_competition_r2(targets_true, preds_post)
    return tot_loss / max(1, samples), r2_raw, r2_post, per_target, preds_raw, preds_post

print("✓ Training and validation engines ready.")

## 8. Execute 5-Fold Training
Runs the complete 2-stage training cycle across all 5 balanced folds and saves checkpoint weights to `models/`.

In [ ]:
oof_preds_post = np.zeros((len(df), 5), dtype=np.float32)
oof_preds_raw = np.zeros((len(df), 5), dtype=np.float32)
oof_targets = df[cfg.target_names].values.astype(np.float32)
fold_scores = []

for fold in range(cfg.n_folds):
    print(f"\n{'='*25} FOLD {fold + 1} / {cfg.n_folds} {'='*25}")
    train_df = df[df['fold'] != fold].reset_index(drop=True)
    val_df = df[df['fold'] == fold].reset_index(drop=True)
    val_indices = df[df['fold'] == fold].index.values
    
    train_loader = DataLoader(
        DualStreamBiomassDataset(train_df, img_size=cfg.img_size, is_training=True),
        batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        DualStreamBiomassDataset(val_df, img_size=cfg.img_size, is_training=False),
        batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True
    )
    
    model = DualStreamDINO(backbone_name=cfg.backbone, pretrained=True).to(device)
    criterion = WeightedBiomassLoss(cls_weight=cfg.cls_weight).to(device)
    scaler = torch.amp.GradScaler('cuda')
    
    best_fold_r2 = -float('inf')
    best_post = None
    best_raw = None
    ckpt_path = os.path.join(cfg.output_dir, f"best_model_fold{fold + 1}.pt")
    
    # STAGE 1: Warmup Heads (Backbone FROZEN)
    print(f"--- [Fold {fold+1}] STAGE 1: Warmup Heads ({cfg.stage1_epochs} epochs) ---")
    for p in model.backbone.parameters():
        p.requires_grad = False
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.stage1_epochs, eta_min=1e-5)
    
    for ep in range(1, cfg.stage1_epochs + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, cfg.grad_accum_steps)
        scheduler.step()
        va_loss, r2_raw, r2_post, _, p_raw, p_post = validate(model, val_loader, criterion, val_df, use_tta=True)
        print(f"[S1 Ep {ep:02d}] Train: {tr_loss:.4f} | Val: {va_loss:.4f} | R2 Raw: {r2_raw:.4f} | R2 Post: {r2_post:.4f}")
        if r2_post > best_fold_r2:
            best_fold_r2 = r2_post
            best_post = p_post
            best_raw = p_raw
            torch.save(model.state_dict(), ckpt_path)
            
    # STAGE 2: Full Fine-Tuning (Backbone UNFROZEN)
    print(f"\n--- [Fold {fold+1}] STAGE 2: Full Fine-Tuning ({cfg.stage2_epochs} epochs) ---")
    for p in model.backbone.parameters():
        p.requires_grad = True
    backbone_lr = cfg.lr * cfg.backbone_lr_ratio
    optimizer = AdamW([
        {'params': model.backbone.parameters(), 'lr': backbone_lr},
        {'params': [p for n, p in model.named_parameters() if not n.startswith('backbone')], 'lr': cfg.lr}
    ], weight_decay=cfg.weight_decay)
    
    warm_sched = LinearLR(optimizer, start_factor=0.1, total_iters=cfg.warmup_epochs)
    cos_sched = CosineAnnealingLR(optimizer, T_max=max(1, cfg.stage2_epochs - cfg.warmup_epochs), eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warm_sched, cos_sched], milestones=[cfg.warmup_epochs])
    
    for ep in range(1, cfg.stage2_epochs + 1):
        curr_ep = cfg.stage1_epochs + ep
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, cfg.grad_accum_steps)
        scheduler.step()
        va_loss, r2_raw, r2_post, per_target, p_raw, p_post = validate(model, val_loader, criterion, val_df, use_tta=True)
        print(f"[S2 Ep {curr_ep:02d}] Train: {tr_loss:.4f} | Val: {va_loss:.4f} | R2 Raw: {r2_raw:.4f} | R2 Post: {r2_post:.4f} | Total: {per_target[4]:.3f} GDM: {per_target[3]:.3f}")
        if r2_post > best_fold_r2:
            best_fold_r2 = r2_post
            best_post = p_post
            best_raw = p_raw
            torch.save(model.state_dict(), ckpt_path)
            print(f"  ★ New Best for Fold {fold+1} (R2 Post: {best_fold_r2:.4f})")
            
    oof_preds_post[val_indices] = best_post
    oof_preds_raw[val_indices] = best_raw
    fold_scores.append(best_fold_r2)
    print(f"✓ Fold {fold+1} Best Post R2: {best_fold_r2:.4f}")

## 9. Final Out-Of-Fold Evaluation & Diagnostics
Computes overall competition score across all 357 held-out validation samples and generates residual plots.

In [ ]:
overall_post_r2, per_target_post = calculate_competition_r2(oof_targets, oof_preds_post)
overall_raw_r2, _ = calculate_competition_r2(oof_targets, oof_preds_raw)

print("=" * 60)
print(f"🏆 FINAL OUT-OF-FOLD (OOF) COMPETITION R²: {overall_post_r2:.4f}")
print(f"  OOF Competition R² (Raw):              {overall_raw_r2:.4f}")
print(f"  Per-Fold Scores: {[round(s, 4) for s in fold_scores]}")
print("  Per-Target Breakdown (Post-Processed):")
for t_name, score, w in zip(cfg.target_names, per_target_post, cfg.official_weights):
    print(f"    - {t_name:15s} (Weight: {w:.1f}): R² = {score:.4f}")
print("=" * 60)

# Save OOF predictions CSV
oof_df = df[['sample_id', 'State', 'Species', 'Sampling_Date'] + cfg.target_names].copy()
for idx, t in enumerate(cfg.target_names):
    oof_df[f'pred_{t}'] = oof_preds_post[:, idx]
oof_csv_path = os.path.join(cfg.output_dir, "oof_predictions.csv")
oof_df.to_csv(oof_csv_path, index=False)
print(f"✓ Saved OOF predictions to {oof_csv_path}")

# Plot Actual vs Predicted for each target
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for i, (ax, t_name) in enumerate(zip(axes, cfg.target_names)):
    y_t = oof_targets[:, i]
    y_p = oof_preds_post[:, i]
    max_val = max(y_t.max(), y_p.max()) * 1.05
    ax.scatter(y_t, y_p, alpha=0.6, edgecolors='none', c='forestgreen')
    ax.plot([0, max_val], [0, max_val], 'r--', lw=1.5)
    ax.set_title(f"{t_name}\nR² = {per_target_post[i]:.3f}", fontsize=11, fontweight='bold')
    ax.set_xlabel("Actual (g)")
    ax.set_ylabel("Predicted (g)")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()